# Le théorème central limite, mesuré — C#

**Navigation** : [version Python](Probas-CLT-Theoreme-Central-Limite.ipynb) · [README Fondations](README.md).

**Probas · Fondations** — jumeau C# de la version Python. Cette expérience mesure le taux de convergence vers la loi normale, puis examine ce qui change lorsque la variance est infinie.

**Objectifs** : calculer une distance de Kolmogorov en tenant compte des deux côtés des sauts de l’ECDF ; distinguer le TCL de la borne de Berry–Esseen et du bruit Monte-Carlo ; mesurer un contre-exemple Lomax sans utiliser une variance inexistante.

**Prérequis** : variables i.i.d., fonction de répartition, moyenne, variance, loi normale, C# et .NET 9. **Durée estimée : 20 minutes**. Les tirages sont réels et les budgets sont ceux de l’expérience Python : 100 000 sommes par cas, puis 60 000 par cas. La graine est 2026 ; les trajectoires ne sont pas identiques entre les langages.

## 1. Mesurer une distance de Kolmogorov

La distance compare les fonctions de répartition. Pour une ECDF, il faut évaluer chaque saut juste avant et juste après la valeur observée. Les contrôles analytiques ci-dessous couvrent un échantillon ponctuel et deux quantiles de la normale.

In [1]:
#r "nuget: MathNet.Numerics, 5.0.0"
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;
using System.Net;
using System.Text;
using MathNet.Numerics.Distributions;
using MathNet.Numerics.Random;
using Microsoft.DotNet.Interactive;
using Microsoft.DotNet.Interactive.Formatting;

CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;
var rng = new Random(2026);
Console.WriteLine($"Runtime .NET {Environment.Version}; MathNet.Numerics 5.0.0; graine 2026; culture invariant.");

Installed Packages MathNet.Numerics, 5.0.0

Runtime .NET 10.0.14; MathNet.Numerics 5.0.0; graine 2026; culture invariant.


### Fonction de mesure et premier exercice

Le code de référence trie les observations et compare l’ECDF aux bornes gauche et droite de chaque saut. Le stub reste vide. Quand une réponse est fournie, les contrôles conditionnels vérifient seulement les résultats analytiques connus.

In [2]:
double Phi(double x) => Normal.CDF(0.0, 1.0, x);

double DistanceKolmogorov(IEnumerable<double> values)
{
    ArgumentNullException.ThrowIfNull(values);
    var xs = values.ToArray();
    if (xs.Length == 0) throw new ArgumentException("L'échantillon doit être non vide.", nameof(values));
    if (xs.Any(x => !double.IsFinite(x)))
        throw new ArgumentException("Toutes les observations doivent être finies.", nameof(values));
    Array.Sort(xs);
    double d = 0.0;
    int i = 0;
    while (i < xs.Length)
    {
        int j = i + 1;
        while (j < xs.Length && xs[j] == xs[i]) j++;
        double cdf = Phi(xs[i]);
        d = Math.Max(d, Math.Abs((double)i / xs.Length - cdf));
        d = Math.Max(d, Math.Abs((double)j / xs.Length - cdf));
        i = j;
    }
    return d;
}

static void VerifierProche(string nom, double observe, double attendu, double tolerance = 1e-7)
{
    if (!double.IsFinite(observe) || Math.Abs(observe - attendu) > tolerance)
        throw new InvalidOperationException($"{nom}: {observe:R} au lieu de {attendu:R}.");
}

static void VerifierRejetContrat<TException>(string nom, Action action) where TException : Exception
{
    try { action(); }
    catch (TException) { return; }
    throw new InvalidOperationException($"Le contrat devait rejeter : {nom}.");
}

VerifierProche("D_K([0])", DistanceKolmogorov(new[] { 0.0 }), 0.5);
const double q75 = 0.6744897501960817;
VerifierProche("D_K des deux quantiles normaux", DistanceKolmogorov(new[] { -q75, q75 }), 0.25, 1e-6);
VerifierProche("D_K de deux sauts égaux et du saut final", DistanceKolmogorov(new[] { -q75, -q75, q75 }), 5.0 / 12.0, 1e-6);
VerifierProche("D_K des répétitions en zéro", DistanceKolmogorov(new[] { 0.0, 0.0, 0.0 }), 0.5);
VerifierRejetContrat<ArgumentException>("ECDF vide", () => DistanceKolmogorov(Array.Empty<double>()));
VerifierRejetContrat<ArgumentException>("ECDF avec NaN", () => DistanceKolmogorov(new[] { double.NaN }));
VerifierRejetContrat<ArgumentException>("ECDF avec infini", () => DistanceKolmogorov(new[] { double.PositiveInfinity }));
Console.WriteLine("Contrôles ECDF réussis : singleton, répétitions, deux côtés des sauts, quantiles et rejet des entrées vides/non finies.");

Contrôles ECDF réussis : singleton, répétitions, deux côtés des sauts, quantiles et rejet des entrées vides/non finies.


Les contrôles analytiques distinguent le saut empirique avant et après une valeur, incluent des répétitions et testent des quantiles normaux. Une réponse constante échoue au cas des quantiles.

### Exercice 1 — calculer la distance de Kolmogorov

**Objectif** : retourner `D_K` entre l’ECDF et `N(0,1)` pour une liste non vide. **Indice** : trier les valeurs, regrouper les répétitions, puis comparer `i/n` et `j/n` à `Φ(x)` pour les comptes strictement inférieur et inférieur ou égal. Les tests conditionnels portent sur le singleton `[0]`, trois zéros et les quantiles `±Φ⁻¹(0,75)`.

In [3]:
// Exercice 1 — implémenter les deux côtés de chaque saut de l'ECDF.
double? KolmogorovEtudiant(IEnumerable<double> values)
{
    // TODO étudiant : trier, puis comparer les comptes avant/après chaque valeur à Phi(x).
    return null;
}

var reponseEx1 = KolmogorovEtudiant(new[] { 0.0 });
if (reponseEx1 is double dEx1)
{
    VerifierProche("Exercice 1, D_K([0])", dEx1, 0.5);
    VerifierProche("Exercice 1, répétitions", KolmogorovEtudiant(new[] { 0.0, 0.0, 0.0 }) ?? double.NaN, 0.5);
    VerifierProche("Exercice 1, deux quantiles", KolmogorovEtudiant(new[] { -q75, q75 }) ?? double.NaN, 0.25, 1e-6);
    VerifierProche("Exercice 1, répétitions mixtes", KolmogorovEtudiant(new[] { -q75, -q75, q75 }) ?? double.NaN, 5.0 / 12.0, 1e-6);
}
else
    Console.WriteLine("Exercice 1 à compléter : les contrôles s'activeront dès qu'une valeur sera renvoyée.");

Exercice 1 à compléter : les contrôles s'activeront dès qu'une valeur sera renvoyée.


## 2. Lois à variance finie : mesure et borne

Le TCL suppose des variables i.i.d. de moyenne finie et de variance strictement positive et finie. Berry–Esseen fournit une borne en `1/√n` lorsque le troisième moment absolu centré est également fini. Cette borne n’est pas une équivalence : une loi normale donne une distance de population nulle à toute taille.

In [4]:
var lois = new (string Nom, double Mu, double Sigma, Func<Random, Func<double>> CreerTireur)[]
{
    ("Uniforme[-1,1]", 0.0, Math.Sqrt(1.0 / 3.0), (Func<Random, Func<double>>)(r => new ContinuousUniform(-1.0, 1.0, r).Sample)),
    ("Bernoulli(0,3)", 0.3, Math.Sqrt(0.3 * 0.7), (Func<Random, Func<double>>)(r => { var d = new Bernoulli(0.3, r); return () => d.Sample(); })),
    ("Exponentielle(1)", 1.0, 1.0, (Func<Random, Func<double>>)(r => new Exponential(1.0, r).Sample)),
    ("Log-normale", Math.Exp(0.5), Math.Sqrt(Math.Exp(2.0) - Math.Exp(1.0)), (Func<Random, Func<double>>)(r => new LogNormal(0.0, 1.0, r).Sample))
};

foreach (var loi in lois)
    Console.WriteLine($"{loi.Nom,-20} μ={loi.Mu:F6}  σ={loi.Sigma:F6}  0<σ²<∞");
Console.WriteLine("Quatre distributions MathNet réelles, avec Random explicite transmis à chaque tirage.");

Uniforme[-1,1]       μ=0.000000  σ=0.577350  0<σ²<∞


Bernoulli(0,3)       μ=0.300000  σ=0.458258  0<σ²<∞


Exponentielle(1)     μ=1.000000  σ=1.000000  0<σ²<∞


Log-normale          μ=1.648721  σ=2.161197  0<σ²<∞


Quatre distributions MathNet réelles, avec Random explicite transmis à chaque tirage.


Les paramètres sont analytiques : `U[-1,1]`, Bernoulli(0,3), exponentielle de taux 1 et `exp(N(0,1))`. Les tirages utilisent les objets de distribution MathNet avec le générateur explicite. Chacune satisfait aussi l’hypothèse de moment absolu d’ordre 3 de Berry–Esseen.

### Sommes standardisées et deux côtés de l’ECDF

Pour chaque loi et `n ∈ {10,40,160}`, on forme 100 000 sommes standardisées. Le calcul est en flux : la mémoire ne dépend pas de `m×n`. Le tableau affiche `D̂_K` et `D̂_K√n`, estimateurs Monte-Carlo de la distance et de son produit.

In [5]:
double[] SommesNormalisees(double mu, double sigma, Func<Random, Func<double>> creerTireur, int n, int m, Random random)
{
    if (n <= 0 || m <= 0 || !(sigma > 0.0) || !double.IsFinite(sigma) || !double.IsFinite(mu))
        throw new ArgumentOutOfRangeException(nameof(n), "n, m et σ doivent être valides ; μ et σ sont finis.");
    var sums = new double[m];
    double denominateur = sigma * Math.Sqrt(n);
    var tirer = creerTireur(random);
    for (int k = 0; k < m; k++)
    {
        double somme = 0.0;
        for (int i = 0; i < n; i++) somme += tirer();
        sums[k] = (somme - n * mu) / denominateur;
    }
    if (sums.Any(x => !double.IsFinite(x)))
        throw new InvalidOperationException("Les sommes normalisées doivent être finies : vérifier les tirages et les paramètres.");
    return sums;
}

VerifierProche("Somme normalisée déterministe", SommesNormalisees(2.0, 3.0, r => () => 5.0, 4, 1, new Random(1))[0], 2.0);
VerifierRejetContrat<ArgumentException>("SommesNormalisees n=0", () => SommesNormalisees(0.0, 1.0, r => () => 0.0, 0, 1, new Random(1)));
VerifierRejetContrat<ArgumentException>("SommesNormalisees m=0", () => SommesNormalisees(0.0, 1.0, r => () => 0.0, 1, 0, new Random(1)));
VerifierRejetContrat<ArgumentException>("SommesNormalisees σ=0", () => SommesNormalisees(0.0, 0.0, r => () => 0.0, 1, 1, new Random(1)));
VerifierRejetContrat<ArgumentException>("SommesNormalisees σ infini", () => SommesNormalisees(0.0, double.PositiveInfinity, r => () => 0.0, 1, 1, new Random(1)));
VerifierRejetContrat<ArgumentException>("SommesNormalisees μ NaN", () => SommesNormalisees(double.NaN, 1.0, r => () => 0.0, 1, 1, new Random(1)));
VerifierRejetContrat<InvalidOperationException>("SommesNormalisees tirage NaN", () => SommesNormalisees(0.0, 1.0, r => () => double.NaN, 1, 1, new Random(1)));
Console.WriteLine("Sommes standardisées : exemple déterministe validé ; n/m/σ/μ invalides et tirage non fini rejetés avant les 84 M tirages.");

const int M = 100_000;
int[] ns = [10, 40, 160];
var mesures = new List<(string Loi, int N, double D, double Produit)>();
foreach (var loi in lois)
foreach (int n in ns)
{
    double d = DistanceKolmogorov(SommesNormalisees(loi.Mu, loi.Sigma, loi.CreerTireur, n, M, rng));
    mesures.Add((loi.Nom, n, d, d * Math.Sqrt(n)));
}
Console.WriteLine($"{M:N0} sommes par loi et taille; {mesures.Count * M:N0} sommes au total; {lois.Length * M * ns.Sum():N0} tirages.");
Console.WriteLine($"{"Loi",-20} {"n",5} {"D_K empirique",16} {"D_K √n",12}");
foreach (var x in mesures)
    Console.WriteLine($"{x.Loi,-20} {x.N,5} {x.D,16:F6} {x.Produit,12:F6}");

Sommes standardisées : exemple déterministe validé ; n/m/σ/μ invalides et tirage non fini rejetés avant les 84 M tirages.


100,000 sommes par loi et taille; 1,200,000 sommes au total; 84,000,000 tirages.


Loi                      n    D_K empirique       D_K √n


Uniforme[-1,1]          10         0.004159     0.013151


Uniforme[-1,1]          40         0.004773     0.030187


Uniforme[-1,1]         160         0.003006     0.038029


... (5 outputs left out)

Exponentielle(1)       160         0.013575     0.171711


Log-normale             10         0.092585     0.292779


Log-normale             40         0.052700     0.333302


Log-normale            160         0.032922     0.416440


Les valeurs sont celles des échantillons, pas les distances exactes entre lois. La théorie autorise une borne `C·E|X−μ|³/(σ³√n)` seulement si le moment absolu d’ordre 3 est fini. Pour `D̂_K√n`, l’erreur absolue Monte-Carlo est d’ordre `√(n/M)` : les valeurs n’ont donc pas à décroître ou à former un plateau parfait à graine fixée. Trois tailles illustrent un ordre de grandeur, elles ne prouvent pas une asymptotique.

### Exercice 2 — estimer la courbe de convergence

**Objectif** : compléter le calcul de `D_K` des sommes standardisées pour les tailles `n={10,40,160}`. **Paramètres** : loi uniforme, `m=2 000` pour cet exercice. **Indice** : construire `m` sommes en flux avec `SommesNormalisees`, puis appeler `DistanceKolmogorov` une fois par taille. Le contrôle conditionnel vérifie uniquement une distance finie entre 0 et 1 par taille.

In [6]:
// Exercice 2 — renvoyer une distance par taille n, sans changer les budgets de l'expérience principale.
double[] CourbeConvergenceEtudiant((string Nom, double Mu, double Sigma, Func<Random, Func<double>> CreerTireur) loi, IReadOnlyList<int> tailles, int m, Random random)
{
    // TODO étudiant : pour chaque n, standardiser m sommes et mesurer DistanceKolmogorov.
    return null;
}

var reponseEx2 = CourbeConvergenceEtudiant(lois[0], ns, 2_000, new Random(31415));
if (reponseEx2 is not null)
{
    if (reponseEx2.Length != ns.Length || reponseEx2.Any(x => !double.IsFinite(x) || x < 0.0 || x > 1.0))
        throw new InvalidOperationException("Exercice 2 : une distance finie par taille, entre 0 et 1, est attendue.");
    Console.WriteLine("Exercice 2 : dimensions et domaine des distances vérifiés.");
}
else Console.WriteLine("Exercice 2 à compléter : la garde validera longueur et domaine lorsqu'une réponse sera fournie.");

Exercice 2 à compléter : la garde validera longueur et domaine lorsqu'une réponse sera fournie.


## 3. Queue lourde : Lomax de variance infinie

La loi de Lomax (Pareto de type II) de forme `α=1,5` et d’échelle `ξ=1` a une moyenne 2 et une variance infinie. MathNet échantillonne ici une vraie Pareto de type I, translatée de 1 : `Pareto(1,1,5) − 1`. Nous mesurons la médiane de `|(Sₙ−nμ)/√n|`, robuste aux extrêmes et définie sans variance.

In [7]:
const double alpha = 1.5;
const double xi = 1.0;
const double muLomax = xi / (alpha - 1.0);
Func<double> CreerLomax(Random random)
{
    var paretoTypeI = new Pareto(xi, alpha, random);
    return () => paretoTypeI.Sample() - xi;
}

double EchelleRobuste(Func<Random, Func<double>> creerTireur, double mu, int n, int m, Random random)
{
    if (n <= 0 || m <= 0 || !double.IsFinite(mu))
        throw new ArgumentOutOfRangeException(nameof(n));
    var ecarts = new double[m];
    var tirer = creerTireur(random);
    for (int k = 0; k < m; k++)
    {
        double somme = 0.0;
        for (int i = 0; i < n; i++) somme += tirer();
        ecarts[k] = Math.Abs((somme - n * mu) / Math.Sqrt(n));
    }
    if (ecarts.Any(x => !double.IsFinite(x)))
        throw new InvalidOperationException("Les écarts robustes doivent être finis : vérifier les tirages et μ.");
    Array.Sort(ecarts);
    return (ecarts[(m - 1) / 2] + ecarts[m / 2]) / 2.0;
}

var ecartsPairControle = new[] { 1.0, 3.0, 5.0, 7.0 };
int indexEcartPairControle = 0;
VerifierProche("Médiane paire des écarts absolus", EchelleRobuste(r => () => ecartsPairControle[indexEcartPairControle++], 0.0, 1, 4, new Random(2)), 4.0);
VerifierRejetContrat<ArgumentException>("EchelleRobuste n=0", () => EchelleRobuste(CreerLomax, muLomax, 0, 1, new Random(2)));
VerifierRejetContrat<ArgumentException>("EchelleRobuste m=0", () => EchelleRobuste(CreerLomax, muLomax, 1, 0, new Random(2)));
VerifierRejetContrat<ArgumentException>("EchelleRobuste μ NaN", () => EchelleRobuste(CreerLomax, double.NaN, 1, 1, new Random(2)));
VerifierRejetContrat<InvalidOperationException>("EchelleRobuste tirage NaN", () => EchelleRobuste(r => () => double.NaN, 0.0, 1, 2, new Random(2)));
Console.WriteLine("Médiane paire validée ; n/m/μ invalides et tirage non fini rejetés avant les 102 M tirages.");
Console.WriteLine($"Lomax: α={alpha:F1}, ξ={xi:F1}, E[X]={muLomax:F1}; Var[X]=∞ car α<2.");
Console.WriteLine("Exemple de tirage Pareto décalé : " + CreerLomax(rng)().ToString("F4"));

Médiane paire validée ; n/m/μ invalides et tirage non fini rejetés avant les 102 M tirages.


Lomax: α=1.5, ξ=1.0, E[X]=2.0; Var[X]=∞ car α<2.


Exemple de tirage Pareto décalé : 0.8554


Pour l’uniforme, le TCL prédit une échelle qui se stabilise près de `Φ⁻¹(0,75)/√3 ≈ 0,3894`. Pour cette Lomax, l’exposant résiduel attendu est `1/α−1/2=1/6`. La croissance prévue à l’asymptote n’exige pas un rapport exact pour des tailles finies.

In [8]:
const int MRobuste = 60_000;
int[] nsRobustes = [10, 40, 160, 640];
var robustes = new List<(int N, double Uniforme, double Lomax)>();
foreach (int n in nsRobustes)
{
    double u = EchelleRobuste(r => new ContinuousUniform(-1.0, 1.0, r).Sample, 0.0, n, MRobuste, rng);
    double l = EchelleRobuste(CreerLomax, muLomax, n, MRobuste, rng);
    robustes.Add((n, u, l));
}
Console.WriteLine($"{MRobuste:N0} sommes par loi et taille; {robustes.Count * 2 * MRobuste:N0} sommes robustes; {MRobuste * nsRobustes.Sum() * 2:N0} tirages.");
Console.WriteLine($"{"n",6} {"Uniforme",14} {"Lomax alpha=1,5",16}");
foreach (var x in robustes)
    Console.WriteLine($"{x.N,6} {x.Uniforme,14:F5} {x.Lomax,16:F5}");
Console.WriteLine($"Limite uniforme: Φ⁻¹(0,75)/√3 ≈ {Normal.InvCDF(0.0, 1.0, 0.75) / Math.Sqrt(3.0):F8}; exposant résiduel Lomax = {1.0 / alpha - 0.5:F6}.");

60,000 sommes par loi et taille; 480,000 sommes robustes; 102,000,000 tirages.


     n       Uniforme  Lomax alpha=1,5


    10        0.39583          2.96347


    40        0.38956          3.97878


   160        0.38703          5.23353


   640        0.39067          6.69745


Limite uniforme: Φ⁻¹(0,75)/√3 ≈ 0.38941684; exposant résiduel Lomax = 0.166667.


La comparaison teste une différence d’échelle empirique. Une hausse de la médiane Lomax est cohérente avec l’exposant `1/6`, mais ne prouve ni la convergence vers une loi stable ni un ratio asymptotique exact. Ce contre-exemple montre que le TCL classique, avec normalisation `√n`, ne s’applique pas à cette loi ; il ne décrit pas à lui seul toutes les lois de variance infinie.

### Exercice 3 — mesurer l’échelle robuste de Lomax

**Objectif** : pour `n = {10,40,160,640}`, calculer la médiane de `|(Sₙ−nμ)/√n|` avec 2 000 répétitions par taille. **Paramètres** : Lomax `α=1,5`, `ξ=1`, `μ=2`. **Indice** : réutiliser `EchelleRobuste` et `CreerLomax`. Le contrôle vérifie longueur et valeurs finies, sans exiger une hausse stricte à cause du bruit fini.

In [9]:
// Exercice 3 — mesurer l'échelle robuste pour des tailles croissantes.
double[] EchelleLomaxEtudiant(IReadOnlyList<int> tailles, int m, Random random)
{
    // TODO étudiant : calculer la médiane des écarts absolus normalisés par √n.
    return null;
}

var reponseEx3 = EchelleLomaxEtudiant(nsRobustes, 2_000, new Random(2718));
if (reponseEx3 is not null)
{
    if (reponseEx3.Length != nsRobustes.Length || reponseEx3.Any(x => !double.IsFinite(x) || x < 0.0))
        throw new InvalidOperationException("Exercice 3 : une échelle finie non négative par taille est attendue.");
    Console.WriteLine("Exercice 3 : dimensions et domaine des échelles vérifiés; aucun ordre strict n'est imposé au bruit fini.");
}
else Console.WriteLine("Exercice 3 à compléter : la garde validera la réponse lorsqu'elle sera fournie.");

Exercice 3 à compléter : la garde validera la réponse lorsqu'elle sera fournie.


## 4. Visualiser les mesures

Les deux figures SVG sont ajoutées dans le jumeau C# : la première compare les distances empiriques et leur produit, la seconde trace les échelles robustes. Les axes et légendes indiquent les quantités observées ; les courbes ne remplacent pas les réserves sur le bruit Monte-Carlo ni les hypothèses des théorèmes.

In [10]:
record PlotSvg(string Content);
Formatter.Register<PlotSvg>((plot, writer) => writer.Write(plot.Content), "text/html");
string Enc(string s) => WebUtility.HtmlEncode(s);
string SvgHeader(int width, int height, string title) => $"<svg xmlns='http://www.w3.org/2000/svg' width='{width}' height='{height}' viewBox='0 0 {width} {height}' role='img' aria-label='{Enc(title)}'><rect width='100%' height='100%' fill='white'/><text x='40' y='30' font-family='sans-serif' font-size='18' font-weight='bold'>{Enc(title)}</text>";

PlotSvg FigureDistances()
{
    const int w = 920, h = 520, left = 80, right = 30, top = 65, panelH = 165, gap = 45;
    double maxD = mesures.Max(x => x.D) * 1.15;
    double maxP = mesures.Max(x => x.Produit) * 1.15;
    var b = new StringBuilder(SvgHeader(w, h, "Distances empiriques à N(0,1) — M = 100 000"));
    var couleurs = new[] { "#1565c0", "#d84315", "#2e7d32", "#8e24aa" };
    void panel(int y0, Func<(string Loi, int N, double D, double Produit), double> valeur, double ymax, string label)
    {
        b.Append($"<line x1='{left}' y1='{y0+panelH}' x2='{w-right}' y2='{y0+panelH}' stroke='#333'/><line x1='{left}' y1='{y0}' x2='{left}' y2='{y0+panelH}' stroke='#333'/><text x='18' y='{y0+panelH/2}' transform='rotate(-90 18 {y0+panelH/2})' font-family='sans-serif' font-size='13'>{Enc(label)}</text>");
        for (int t=0;t<=4;t++) { double v=ymax*t/4; int yy=y0+panelH-(int)(panelH*t/4.0); b.Append($"<line x1='{left-4}' y1='{yy}' x2='{w-right}' y2='{yy}' stroke='#ddd'/><text x='{left-10}' y='{yy+4}' text-anchor='end' font-family='sans-serif' font-size='11'>{v:F3}</text>"); }
        for (int li=0;li<lois.Length;li++) { var serie=mesures.Where(x=>x.Loi==lois[li].Nom).OrderBy(x=>x.N).ToArray(); var pts=serie.Select((x,i)=>$"{left+60+i*220},{y0+panelH-(int)(panelH*valeur(x)/ymax)}"); b.Append($"<polyline points='{string.Join(" ",pts)}' fill='none' stroke='{couleurs[li]}' stroke-width='2.5'/>"); foreach(var x in serie) { int xx=left+60+Array.IndexOf(ns,x.N)*220, yy=y0+panelH-(int)(panelH*valeur(x)/ymax); b.Append($"<circle cx='{xx}' cy='{yy}' r='4' fill='{couleurs[li]}'/>"); } }
        for(int i=0;i<ns.Length;i++) b.Append($"<text x='{left+60+i*220}' y='{y0+panelH+19}' text-anchor='middle' font-family='sans-serif' font-size='12'>n={ns[i]}</text>");
    }
    panel(top, x=>x.D, maxD, "D̂_K empirique");
    panel(top+panelH+gap, x=>x.Produit, maxP, "D̂_K √n");
    for(int li=0;li<lois.Length;li++) { int x=left+li*200; b.Append($"<line x1='{x}' y1='492' x2='{x+22}' y2='492' stroke='{couleurs[li]}' stroke-width='3'/><text x='{x+28}' y='496' font-family='sans-serif' font-size='12'>{Enc(lois[li].Nom)}</text>"); }
    b.Append("</svg>"); return new PlotSvg(b.ToString());
}

display(FigureDistances());
Console.WriteLine("Figure SVG construite depuis les 12 mesures exécutées; fond blanc, axes, légendes et titres échappés pour XML.");

Distances empiriques à N(0,1) — M = 100 000 D̂_K empirique 0.000 0.043 0.086 0.129 0.171 n=10 n=40 n=160 D̂_K √n 0.000 0.153 0.306 0.459 0.612 n=10 n=40 n=160 Uniforme[-1,1] Bernoulli(0,3) Exponentielle(1) Log-normale

Figure SVG construite depuis les 12 mesures exécutées; fond blanc, axes, légendes et titres échappés pour XML.


### Échelles robustes comparées

La seconde figure place les médianes uniforme et Lomax sur les mêmes axes. La valeur limite théorique de l’uniforme sert de repère textuel ; aucune série asymptotique n’est simulée.

In [11]:
PlotSvg FigureEchelles()
{
    const int w = 780, h = 470, left = 88, right = 35, top = 60, bottom = 100;
    double ymax = robustes.Max(x => x.Lomax) * 1.15;
    var b = new StringBuilder(SvgHeader(w, h, "Médiane de |(S_n - n*mu)/sqrt(n)| — M = 60 000"));
    int pw=w-left-right, ph=h-top-bottom;
    b.Append($"<line x1='{left}' y1='{top+ph}' x2='{w-right}' y2='{top+ph}' stroke='#333'/><line x1='{left}' y1='{top}' x2='{left}' y2='{top+ph}' stroke='#333'/>");
    for(int t=0;t<=5;t++){ double v=ymax*t/5; int y=top+ph-(int)(ph*t/5.0); b.Append($"<line x1='{left}' y1='{y}' x2='{w-right}' y2='{y}' stroke='#ddd'/><text x='{left-10}' y='{y+4}' text-anchor='end' font-family='sans-serif' font-size='11'>{v:F1}</text>"); }
    double X(int i)=>left+(double)i*pw/(nsRobustes.Length-1); double Y(double v)=>top+ph-v/ymax*ph;
    foreach(var (nom, val, couleur) in new[]{("Uniforme",(Func<(int N,double Uniforme,double Lomax),double>)(x=>x.Uniforme),"#1565c0"),("Lomax alpha=1,5",(Func<(int N,double Uniforme,double Lomax),double>)(x=>x.Lomax),"#d84315")})
    {
        b.Append($"<polyline points='{string.Join(" ",robustes.Select((x,i)=>$"{X(i):F1},{Y(val(x)):F1}"))}' fill='none' stroke='{couleur}' stroke-width='3'/>");
        for(int i=0;i<robustes.Count;i++) b.Append($"<circle cx='{X(i):F1}' cy='{Y(val(robustes[i])):F1}' r='5' fill='{couleur}'/>");
        int lx=nom=="Uniforme"?left+15:left+190; b.Append($"<line x1='{lx}' y1='430' x2='{lx+22}' y2='430' stroke='{couleur}' stroke-width='3'/><text x='{lx+28}' y='434' font-family='sans-serif' font-size='13'>{Enc(nom)}</text>");
    }
    for(int i=0;i<nsRobustes.Length;i++) b.Append($"<text x='{X(i):F1}' y='{top+ph+24}' text-anchor='middle' font-family='sans-serif' font-size='12'>n={nsRobustes[i]}</text>");
    b.Append($"<text x='18' y='{top+ph/2}' transform='rotate(-90 18 {top+ph/2})' font-family='sans-serif' font-size='13'>Médiane de |z_n|</text><text x='{left+pw/2}' y='{h-15}' text-anchor='middle' font-family='sans-serif' font-size='13'>Taille de la somme n</text></svg>");
    return new PlotSvg(b.ToString());
}

display(FigureEchelles());
Console.WriteLine("Figure SVG construite depuis les huit médianes observées; le pointillé asymptotique n'est pas simulé.");

Médiane de |(S_n - n*mu)/sqrt(n)| — M = 60 000 0.0 1.5 3.1 4.6 6.2 7.7 Uniforme Lomax alpha=1,5 n=10 n=40 n=160 n=640 Médiane de |z_n| Taille de la somme n

Figure SVG construite depuis les huit médianes observées; le pointillé asymptotique n'est pas simulé.


L’uniforme est confrontée à sa limite analytique, tandis que la courbe Lomax reste une mesure à taille finie. Ni la seule pente visuelle ni quatre points ne suffisent à établir une loi limite.

## À retenir

| Question | Réponse mesurée ou théorique |
|---|---|
| Quelle hypothèse porte le TCL classique ? | Des variables i.i.d. de variance finie et strictement positive ; il s’agit d’une condition suffisante, pas d’une caractérisation complète. |
| Que donne Berry–Esseen ? | Une majoration en `1/√n` si le troisième moment absolu centré est fini, pas une équivalence ni une constante asymptotique positive. |
| Que mesure le tableau ? | Une distance empirique à la normale, affectée par le bruit Monte-Carlo ; `D̂_K√n` ne doit pas être parfaitement constant. |
| Pourquoi la Lomax est-elle un contre-exemple ? | Sa variance est infinie ; la normalisation gaussienne `√n` ne stabilise pas son échelle robuste dans l’expérience. |

Les figures de ce jumeau sont des visualisations ajoutées. Toutes les conclusions quantitatives doivent se lire dans les sorties exécutées ci-dessus.

## Vérifiez votre compréhension

**1. Pourquoi compare-t-on les deux côtés de chaque saut de l’ECDF ?**

<details><summary>Réponse</summary>

La fonction empirique saute au point observé : sa valeur juste avant compte les observations strictement inférieures, et sa valeur après compte aussi les répétitions égales. La plus grande différence à Φ peut se trouver de chaque côté.

</details>

**2. Une petite distance empirique pour `n=160` prouve-t-elle une vitesse `1/√n` ?**

<details><summary>Réponse</summary>

Non. C’est une estimation sur un échantillon fini. Berry–Esseen donne une borne sous hypothèse de troisième moment, et le bruit Monte-Carlo sur le produit amplifie avec `√n` à budget `M` fixé. Quelques tailles ne prouvent pas une asymptotique.

</details>

**3. Pourquoi ne calcule-t-on pas la variance des sommes Lomax ?**

<details><summary>Réponse</summary>

Pour `α=1,5`, la variance théorique n’existe pas. Une variance empirique serait dominée par des extrêmes et ne serait pas un estimateur stable de paramètre fini. La médiane de l’écart absolu sert ici d’échelle robuste.

</details>

## Pour aller plus loin

- [Index de la série Probas](../README.md) : parcours des fondations et applications.
- [Version Python du TCL mesuré](Probas-CLT-Theoreme-Central-Limite.ipynb) : même protocole, avec NumPy.
- [La loi stable](https://en.wikipedia.org/wiki/Stable_distribution) : comprendre les limites possibles des sommes à queue lourde, au-delà du TCL classique.